# 03 — Per-Image PCA Merged Clustering

This notebook loads the five per-image-type PCA feature matrices produced by
`02b_pca_per_image_new_method.ipynb`, merges them on `patient_id` into a single
wide matrix, and saves it for downstream clustering experiments.

## Step 1 — Load the five padded PCA feature files

The CSVs were saved by `02b_pca_per_image_new_method.ipynb` to the main
`notebooks/` directory.  Each file contains one row per patient (416 rows)
and 110 PCA components prefixed with the image-type label.

In [2]:
import os
import pandas as pd
from functools import reduce

DATA_DIR = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data"
NB_DIR   = os.path.join(DATA_DIR, "notebooks")

pca_files = {
    "cor":        "pca_cor.csv",
    "gfc_sag":    "pca_gfc_sag.csv",
    "gfc_tra":    "pca_gfc_tra.csv",
    "masked_tra": "pca_masked_tra.csv",
    "sbj_sag":    "pca_sbj_sag.csv",
}

dfs = []
for label, fname in pca_files.items():
    path = os.path.join(NB_DIR, fname)
    df   = pd.read_csv(path)
    print(f"{label:12s}: {df.shape}  columns: patient_id + {df.shape[1]-1} PCs")
    dfs.append(df)

cor         : (416, 111)  columns: patient_id + 110 PCs
gfc_sag     : (416, 111)  columns: patient_id + 110 PCs
gfc_tra     : (416, 111)  columns: patient_id + 110 PCs
masked_tra  : (416, 111)  columns: patient_id + 110 PCs
sbj_sag     : (416, 111)  columns: patient_id + 110 PCs


## Step 2 — Merge all five files on `patient_id`

An inner join ensures only patients present in every file are kept
(all 5 image types must be available).

In [3]:
merged = reduce(lambda l, r: l.merge(r, on="patient_id", how="inner"), dfs)

print(f"Merged matrix shape : {merged.shape}")
print(f"  rows              : {merged.shape[0]}  (patients)")
print(f"  cols              : {merged.shape[1]}  (patient_id + {merged.shape[1]-1} PCA features)")

Merged matrix shape : (416, 551)
  rows              : 416  (patients)
  cols              : 551  (patient_id + 550 PCA features)


## Step 3 — Preview

First 3 rows and first 10 columns of the merged matrix.

In [4]:
merged.iloc[:3, :10]

,patient_id,cor_PC1,cor_PC2,cor_PC3,cor_PC4,cor_PC5,cor_PC6,cor_PC7,cor_PC8,cor_PC9
0,OAS1_0001,-2.733609,-0.262091,-0.753396,-0.805684,0.235622,0.087258,1.304143,-0.004402,-0.316841
1,OAS1_0002,1.980450,0.446801,-1.209307,0.267108,0.040703,-0.450104,0.901845,0.682579,0.391760
2,OAS1_0003,-1.515047,-1.209053,-0.827747,1.025572,0.436428,1.150057,0.239044,-0.026326,0.091560


## Step 4 — Save the merged matrix

In [5]:
OUT_CSV = os.path.join(NB_DIR, "pca_all_images_merged.csv")
merged.to_csv(OUT_CSV, index=False)
print(f"Saved merged matrix to: {OUT_CSV}")

Saved merged matrix to: C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\pca_all_images_merged.csv
